# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
spaceship.shape

(8693, 14)

In [4]:
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [5]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [6]:
spaceship_clean = spaceship.dropna().copy()

spaceship_clean.isna().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [7]:
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str.split("/").str[0]

spaceship_clean["Cabin"].value_counts()

Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

In [8]:
spaceship_clean = spaceship_clean.drop(columns=["PassengerId", "Name"])

spaceship_clean.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [9]:
spaceship_encoded = pd.get_dummies(spaceship_clean, drop_first=True)

spaceship_encoded.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False


In [10]:
X = spaceship_encoded.drop(columns=["Transported"])
y = spaceship_encoded["Transported"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6606, 19)
y shape: (6606,)


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [12]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

X_scaled.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.695413,-0.345756,-0.285355,-0.309494,-0.273759,-0.269534,1.717147,-0.510811,-0.738664,3.085305,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
1,-0.336769,-0.176748,-0.279993,-0.266112,0.206165,-0.230494,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
2,2.002842,-0.279083,1.845163,-0.309494,5.596357,-0.226058,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,6.306963
3,0.282540,-0.345756,0.479034,0.334285,2.636384,-0.098291,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
4,-0.887266,0.124056,-0.243650,-0.047470,0.220152,-0.267759,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555


In [13]:
selector = SelectKBest(score_func=f_classif, k=20)

X_selected = selector.fit_transform(X_scaled, y)

selected_features = X_scaled.columns[selector.get_support()]

X_selected = pd.DataFrame(
    X_selected,
    columns=selected_features,
    index=X_scaled.index
)

X_selected.head()

/Users/gabrielbohorquez/Library/Python/3.9/lib/python/site-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=20 is greater than n_features=19. All the features will be returned.
  warnings.warn(


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.695413,-0.345756,-0.285355,-0.309494,-0.273759,-0.269534,1.717147,-0.510811,-0.738664,3.085305,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
1,-0.336769,-0.176748,-0.279993,-0.266112,0.206165,-0.230494,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
2,2.002842,-0.279083,1.845163,-0.309494,5.596357,-0.226058,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,6.306963
3,0.282540,-0.345756,0.479034,0.334285,2.636384,-0.098291,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
4,-0.887266,0.124056,-0.243650,-0.047470,0.220152,-0.267759,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555


In [14]:
selected_features

Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'HomePlanet_Europa', 'HomePlanet_Mars', 'CryoSleep_True', 'Cabin_B',
       'Cabin_C', 'Cabin_D', 'Cabin_E', 'Cabin_F', 'Cabin_G', 'Cabin_T',
       'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e', 'VIP_True'],
      dtype='object')

**Perform Train Test Split**

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5284, 19)
X_test: (1322, 19)
y_train: (5284,)
y_test: (1322,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [16]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    bootstrap=True,
    random_state=42
)

bagging_model.fit(X_train, y_train)

bagging_pred = bagging_model.predict(X_test)

bagging_accuracy = accuracy_score(y_test, bagging_pred)

print(f"Bagging Accuracy: {bagging_accuracy:.4f}")

Bagging Accuracy: 0.7753


In [17]:
pasting_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    bootstrap=False,
    random_state=42
)

pasting_model.fit(X_train, y_train)

pasting_pred = pasting_model.predict(X_test)

pasting_accuracy = accuracy_score(y_test, pasting_pred)

print(f"Pasting Accuracy: {pasting_accuracy:.4f}")

Pasting Accuracy: 0.7466


- Random Forests

In [18]:
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

random_forest_model.fit(X_train, y_train)

random_forest_pred = random_forest_model.predict(X_test)

random_forest_accuracy = accuracy_score(y_test, random_forest_pred)

print(f"Random Forest Accuracy: {random_forest_accuracy:.4f}")

Random Forest Accuracy: 0.7806


- Gradient Boosting

In [19]:
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gradient_boosting_model.fit(X_train, y_train)

gradient_boosting_pred = gradient_boosting_model.predict(X_test)

gradient_boosting_accuracy = accuracy_score(y_test, gradient_boosting_pred)

print(f"Gradient Boosting Accuracy: {gradient_boosting_accuracy:.4f}")

Gradient Boosting Accuracy: 0.7837


- Adaptive Boosting

In [20]:
adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

adaboost_model.fit(X_train, y_train)

adaboost_pred = adaboost_model.predict(X_test)

adaboost_accuracy = accuracy_score(y_test, adaboost_pred)

print(f"AdaBoost Accuracy: {adaboost_accuracy:.4f}")

AdaBoost Accuracy: 0.7542


Which model is the best and why?

In [21]:
model_results = pd.DataFrame({
    "Model": [
        "Bagging",
        "Pasting",
        "Random Forest",
        "Gradient Boosting",
        "AdaBoost"
    ],
    "Accuracy": [
        bagging_accuracy,
        pasting_accuracy,
        random_forest_accuracy,
        gradient_boosting_accuracy,
        adaboost_accuracy
    ]
})

model_results = model_results.sort_values(by="Accuracy", ascending=False)

model_results

,Model,Accuracy
3,Gradient Boosting,0.783661
2,Random Forest,0.780635
0,Bagging,0.775340
4,AdaBoost,0.754160
1,Pasting,0.746596


In [22]:
best_model = model_results.iloc[0]

print("Best model:")
print(best_model)

Best model:
Model       Gradient Boosting
Accuracy             0.783661
Name: 3, dtype: object


In [23]:
predictions_dict = {
    "Bagging": bagging_pred,
    "Pasting": pasting_pred,
    "Random Forest": random_forest_pred,
    "Gradient Boosting": gradient_boosting_pred,
    "AdaBoost": adaboost_pred
}

best_model_name = best_model["Model"]
best_predictions = predictions_dict[best_model_name]

print(f"Classification report for {best_model_name}:")
print(classification_report(y_test, best_predictions))

Classification report for Gradient Boosting:
              precision    recall  f1-score   support

       False       0.82      0.72      0.77       656
        True       0.75      0.85      0.80       666

    accuracy                           0.78      1322
   macro avg       0.79      0.78      0.78      1322
weighted avg       0.79      0.78      0.78      1322



In [24]:
confusion_matrix(y_test, best_predictions)

array([[471, 185],
       [101, 565]])

## Best Model Conclusion

The best model is the one with the highest accuracy in the comparison table.

In this analysis, the best-performing model was selected based on accuracy on the test set.

Ensemble methods usually perform better than a single basic model because they combine multiple learners and reduce individual model errors.

If Random Forest or Gradient Boosting achieved the best score, this suggests that tree-based ensemble methods are better suited for this dataset than simpler approaches. These models can capture non-linear relationships and interactions between features more effectively.

However, the final model should not be selected only by accuracy. It is also important to consider precision, recall, F1-score, interpretability, and business objective.

## Final Conclusion

In this lab, several ensemble methods were tested to improve the previous predictive model.

The dataset was prepared using the same feature engineering approach as before: missing values were removed, `Cabin` was simplified, irrelevant identifier columns were dropped, and categorical variables were converted into dummy variables.

After applying scaling and feature selection, different ensemble models were trained and evaluated: Bagging, Pasting, Random Forest, Gradient Boosting, and AdaBoost.

The best model was selected based on test accuracy. Ensemble models are useful because they combine multiple learners and often produce more stable and accurate predictions than a single model.

However, model selection should not rely only on accuracy. Precision, recall, F1-score, interpretability, and business context should also be considered.